# Batch Convert PDBQT to SDF

This script converts all `.pdbqt` files from a specified input folder into `.sdf` format (recommended for Biovia) and saves them to an output folder.

In [2]:
import os
from openbabel import pybel
from pathlib import Path

def batch_convert_pdbqt(input_folder, output_folder, output_format="sdf"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"📁 Created results folder: {output_folder}")

    files = [f for f in os.listdir(input_folder) if f.endswith(".pdbqt")]
    
    if not files:
        print("❌ No .pdbqt files found.")
        return

    print(f"Found {len(files)} files. Starting conversion to {output_format.upper()}...")
    
    count = 0
    
    for filename in files:
        input_path = os.path.join(input_folder, filename)
        
        name_without_ext = os.path.splitext(filename)[0]
        output_filename = f"{name_without_ext}.{output_format}"
        output_path = os.path.join(output_folder, output_filename)
        
        molecules = list(pybel.readfile("pdbqt", input_path))
        
        if not molecules:
            print(f"⚠️ Empty or corrupt file: {filename}")
            continue

        output_file = pybel.Outputfile(output_format, output_path, overwrite=True)
        
        for mol in molecules:
            if not mol.title:
                mol.title = name_without_ext
            
            output_file.write(mol)
            
        output_file.close()
        
        count += 1
        print(f"✅ {filename} -> {output_filename} ({len(molecules)} poses)")

    print(f"\n🎉 Done! Processed {count} files.")

## Configuration and Execution

In [ ]:
if __name__ == "__main__":
    INPUT_DIR = Path("..") / "data" / "docking" / "run_2025-12-16" / "output"
    OUTPUT_DIR = Path("..") / "data" / "docking" / "run_2025-12-16" / "output_sdf"
    TARGET_FORMAT = "sdf" 

    batch_convert_pdbqt(INPUT_DIR, OUTPUT_DIR, TARGET_FORMAT)

# Prepare for LigPlot+

Analyzes the receptor and ligand to create a complex PDB file suitable for LigPlot+ analysis.
1. Reads the receptor PDBQT, extracting coordinates and formatting as PDB.
2. Reads the ligand PDBQT, converting it to PDB format and ensuring proper residue naming (LIG) and chain ID (Z).

In [ ]:
def prepare_for_ligplot(receptor_path, ligand_path, output_path):
    print(f"🔧 Creating complex for LigPlot+...")
    
    receptor_lines = []
    ligand_lines = []

    # --- RECEPTOR ---
    print(f"   Reading receptor: {receptor_path}")
    try:
        with open(receptor_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith(("ROOT", "BRANCH", "TORSDOF", "ENDROOT")):
                    continue
                
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    clean_line = line[:66]
                    if len(clean_line) < 66:
                        clean_line = clean_line.ljust(66)
                    
                    full_line = f"{clean_line}  1.00  0.00           \n"
                    receptor_lines.append(full_line)
                    
        print(f"✅ Receptor read: {len(receptor_lines)} atoms.")
    except Exception as e:
        print(f"❌ Receptor error: {e}")
        return

    # --- LIGAND ---
    lig_res_name = "LIG"
    lig_chain = "Z"
    lig_res_num = 1
    
    print(f"   Reading ligand: {ligand_path}")
    try:
        mols = list(pybel.readfile("pdbqt", ligand_path))
        if not mols: 
            raise Exception("Ligand file is empty or invalid format.")
        
        ligand_mol = mols[0]
        pdb_block = ligand_mol.write("pdb")
        
        atom_counter = 0
        
        for line in pdb_block.splitlines():
            if line.startswith("ATOM") or line.startswith("HETATM"):
                atom_counter += 1
                try:
                    serial = line[6:11]
                    atom_name = line[12:16]
                    x = line[30:38]
                    y = line[38:46]
                    z = line[46:54]
                    element = line[76:78].strip()
                    
                    new_line = "HETATM{:>5} {:<4} {:>3} {:1}{:>4}    {}{}{}  1.00  0.00           {:>2}\n".format(
                        atom_counter,
                        atom_name,
                        lig_res_name,
                        lig_chain,
                        lig_res_num,
                        x, y, z,
                        element
                    )
                    ligand_lines.append(new_line)
                except IndexError:
                    line = "HETATM" + line[6:] + "\n"
                    ligand_lines.append(line)
                    
            elif line.startswith("CONECT"):
                ligand_lines.append(line + "\n")
                
        print(f"✅ Ligand processed: {atom_counter} atoms (Chain {lig_chain}, Res {lig_res_name}).")

    except Exception as e:
        print(f"❌ Ligand error: {e}")
        import traceback
        traceback.print_exc()
        return

    # --- WRITE ---
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("REMARK    Generated by Python Script for LigPlot+\n")
            f.writelines(receptor_lines)
            f.write("TER\n")
            f.writelines(ligand_lines)
            f.write("END\n")
        
        abs_path = os.path.abspath(output_path)
        print(f"\n🎉 File ready: {output_path}")
        print("-" * 60)
        print("📋 LIGPLOT DATA:")
        print(f"   Residue Name: {lig_res_name}")
        print(f"   Residue Num:  {lig_res_num}")
        print(f"   Chain ID:     {lig_chain}")
        print("-" * 60)
        
        print("🚀 Launch command (in terminal):")
        print(f'ligplot "{abs_path}" {lig_res_name} {lig_res_num} {lig_chain}')
        
    except Exception as e:
        print(f"❌ Write error: {e}")

In [ ]:
# Example Usage
if __name__ == "__main__":
    REC = "molecules/3FXI.pdbqt"
    LIG = "docking_results/ligand_0_H_H_H_SO2NH2_out.pdbqt" 
    OUT = "complex_for_ligplot.pdb"

    if not os.path.exists(REC):
        print(f"❌ Receptor not found: {REC}")
    elif not os.path.exists(LIG):
        print(f"❌ Ligand not found: {LIG}")
    else:
        prepare_for_ligplot(REC, LIG, OUT)